In [12]:
import numpy as np
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import tensorflow as tf
import keras
from keras import layers, Sequential

import keras 
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from keras.callbacks import EarlyStopping, ModelCheckpoint
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
import tarfile
dataset_path = '/kaggle/input/datasets/supreethrao/chars74kdigitalenglishfont/EnglishFnt.tgz'
with tarfile.open(dataset_path, "r:gz") as tar:
    tar.extractall(path="/kaggle/working/extracted_tgz")
    print("Files extracted successfully.")

/tmp/ipykernel_57/2729173537.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="/kaggle/working/extracted_tgz")


Files extracted successfully.


In [3]:
def one_hot_label(label):
    return F.one_hot(torch.tensor(label), num_classes=62).float()
    
dataset_path = r'/kaggle/working/extracted_tgz/English/Fnt'

train_ds = keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(32, 32),
    batch_size=32,
    color_mode='grayscale',
    label_mode='categorical'
)
val_ds = keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(32, 32),
    batch_size=32,
    color_mode='grayscale',
    label_mode='categorical'
)
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

Found 62992 files belonging to 62 classes.
Using 50394 files for training.


I0000 00:00:1778257900.758320      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 62992 files belonging to 62 classes.
Using 12598 files for validation.


In [4]:
model = Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), 
    layers.Dense(62, activation='softmax')
])

model.compile(
    optimizer='adam', 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 62)             │         7,998 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 166,334 (649.74 KB)

 Trainable params: 166,334 (649.74 KB)

...

Epoch 1/10


I0000 00:00:1778257907.261184     135 service.cc:152] XLA service 0x792f64005d40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778257907.261439     135 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1778257907.672737     135 cuda_dnn.cc:529] Loaded cuDNN version 91002


  36/1575 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.0174 - loss: 4.1286

I0000 00:00:1778257910.417753     135 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1575/1575 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.4245 - loss: 2.1940 - val_accuracy: 0.8152 - val_loss: 0.5625
Epoch 2/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7727 - loss: 0.6954 - val_accuracy: 0.8420 - val_loss: 0.4564
Epoch 3/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8069 - loss: 0.5633 - val_accuracy: 0.8537 - val_loss: 0.4151
Epoch 4/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8228 - loss: 0.4952 - val_accuracy: 0.8598 - val_loss: 0.3753
Epoch 5/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8377 - loss: 0.4443 - val_accuracy: 0.8674 - val_loss: 0.3594
Epoch 6/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8433 - loss: 0.4160 - val_accuracy: 0.8709 - val_loss: 0.3440
Epoch 7/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8523 - loss: 0.3856 - val_accuracy: 0.8754 - val_loss: 0.3337
Epoch 8/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8605 - loss: 0.3611 - val_accura

In [13]:
callbacks = [
    EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True),
    ModelCheckpoint(filepath='best_model.keras', monitor='val_loss', save_best_only=True)
]
model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

Epoch 1/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9023 - loss: 0.2372 - val_accuracy: 0.8951 - val_loss: 0.2970
Epoch 2/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9065 - loss: 0.2304 - val_accuracy: 0.8960 - val_loss: 0.2830
Epoch 3/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9101 - loss: 0.2266 - val_accuracy: 0.8943 - val_loss: 0.3044
Epoch 4/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9092 - loss: 0.2203 - val_accuracy: 0.8971 - val_loss: 0.2942
Epoch 5/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9135 - loss: 0.2152 - val_accuracy: 0.8942 - val_loss: 0.3114
Epoch 6/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9121 - loss: 0.2142 - val_accuracy: 0.8993 - val_loss: 0.2898
Epoch 7/10
1575/1575 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9122 - loss: 0.2154 - val_accuracy: 0.8938 - val_loss: 0.3096


In [18]:
model.export("saved_model_dir")

INFO:tensorflow:Assets written to: saved_model_dir/assets


INFO:tensorflow:Assets written to: saved_model_dir/assets


Saved artifact at 'saved_model_dir'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 62), dtype=tf.float32, name=None)
Captures:
  133245664613712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133245632431888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133245632430352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133245632434384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133245632432272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133245632431504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133245632434000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133245632434192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133245632432656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133245632435344: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [19]:
!python -m tf2onnx.convert --saved-model saved_model_dir --output model.onnx --opset 13

E0000 00:00:1778258587.196835    1054 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778258587.203978    1054 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778258587.225135    1054 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778258587.225189    1054 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778258587.225202    1054 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778258587.225212    1054 computation_placer.cc:177] computation placer already registered. Please check linka